# 🌌 APEX Launcher

Welcome to the **APEX** runtime environment. This notebook is a minimal loader that delegates all execution, dependencies, and synchronization to the `APEX Bootstrap Manager`.

In [ ]:
import sys
import subprocess
from pathlib import Path

# 1. Environment Detection & Storage
is_colab = "google.colab" in sys.modules
if is_colab:
    print("[INFO] Google Colab detected")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[SUCCESS] Drive Mounted")
    except Exception:
        print("[WARNING] Drive mount failed. Using temporary storage.")

# 2. Stage-0 Initial Clone (Required to load APEX managers)
repo_url = "https://github.com/Nikhil-Kumar-Shah/APEX.git"
target_dir = Path("/content/APEX") if is_colab else Path.cwd().parent / "APEX"

if not (target_dir / ".git").exists():
    print("[INFO] Fetching APEX Runtime...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", repo_url, str(target_dir)], check=True)

if str(target_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(target_dir.resolve()))

# 3. Build Runtime Context & Delegate to Bootstrap Orchestrator
from bootstrap.manager import RuntimeContext, BootstrapManager

context = RuntimeContext(
    environment="colab" if is_colab else "local",
    persistence=Path("/content/drive/MyDrive/APEX") if is_colab else target_dir.parent / "APEX_Data",
    runtime=target_dir
)

print("[INFO] Starting APEX... Logging will now redirect to the Runtime Console.")
BootstrapManager(context).launch()
